# Banyan City — free Wan T2V rendering on Kaggle
Renders a node's `shots.md` prompts into per-beat clips with **open Apache-2.0
[Wan 2.1 T2V 1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers) weights** on Kaggle's
free GPU quota (30 h/week) — the tree's permanent $0 rendering floor, reproducible by any citizen
(**compute-as-watering**, see `WATERING.md`).

**Setup:** Kaggle → New Notebook → File → Import Notebook → this file. Settings: Accelerator = **GPU
T4 x2**, Internet = ON, phone-verified account. **Run All** (no kernel restart needed — Kaggle's own torch is left untouched). ~20-45 min per 5s clip. Since loop cycle 007 an episode is 15-25 SHOTS (one per beat, 3-6s each,
camera on the referent — see SCRIPT-SPEC.md), so a full episode is ~8-15 GPU-hours: about one
episode per week inside the free 30 h quota. The notebook skips clips that already exist, so a
preempted session resumes by re-running. Output: `/kaggle/working/clips.zip` → feed to
`pipeline/render_t3.py <genome> <node> --clips <dir>`.

Provenance: every clip gets a `meta.yaml` (§7.2). Output is 480×832 (9:16) 5s — the current best a
free T4 does; the season's canon quality bar is decided by the founder on material (R4/D8).

**Free-tier notes (learned on the first real run, 2026-07-25):** Kaggle's free instance is
RAM-poor (~13 GB) and VRAM-ok (T4 = 15 GB), so the 1.3B model is kept **resident on the GPU** —
`enable_model_cpu_offload()` streams weights through system RAM and kills the kernel. Torch is NEVER reinstalled (a fresh torchvision against the running torch
raises `INTERNAL ASSERT FAILED` in `Dtype.cpp`); diffusers goes in with `--no-deps` at >= 0.33,
which is the first version containing `WanPipeline`.
If the kernel still dies, drop `STEPS` to 30 and re-run: finished clips are skipped.


In [ ]:
# ---- config: what to render ----------------------------------------------
GENOME = "sapling"
NODE   = "002b"        # any node id with a shots.md
BEATS  = None          # e.g. [1, 3] or None for all beats without status ✅
SEED   = 20260719      # fixed base seed: beat N renders with SEED + N (reproducible)
STEPS  = 40            # 30 = faster/rougher, 50 = slower/cleaner
REPO_URL = "https://github.com/olegmlkvorg/banyan-city.git"


In [ ]:
# ---- setup: deps + repo (canon prompts come from shots.md, not a paste) ---
# Do NOT reinstall torch. Kaggle ships a working torch/torchvision pair; the
# first real run (2026-07-25) tried to pin its own and hit INTERNAL ASSERT
# FAILED in Dtype.cpp — a fresh torchvision against the already-imported
# torch. Install diffusers with --no-deps so pip cannot pull a second torch in
# behind it. WanPipeline needs diffusers >= 0.33 (0.32 lacks it entirely —
# that was the failure after the pin).
%pip -q install --no-deps "diffusers==0.33.1"
%pip -q install ftfy imageio imageio-ffmpeg pyyaml

import pathlib
import subprocess
import sys

if not pathlib.Path("banyan-city").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
sys.path.insert(0, "banyan-city/pipeline")
from generate_shots import parse_shots
import yaml

node_dirs = [d for d in pathlib.Path(f"banyan-city/genomes/{GENOME}/nodes").iterdir() if d.is_dir()]
node_dir = next((d for d in sorted(node_dirs) if d.name.startswith(NODE)), None)
assert node_dir, f"no node dir starting with {NODE!r} — check NODE above"
shots = parse_shots((node_dir / "shots.md").read_text())
todo = [s for s in shots if (BEATS is None and not s["done"]) or (BEATS and s["num"] in BEATS)]
print(f"{len(todo)} beat(s) to render for {node_dir.name}:")
for s in todo:
    print(f"  {s['num']:02d} {s['slug']}")


In [ ]:
# ---- model: Wan 2.1 T2V 1.3B (Apache-2.0) ----------------------------------
# Kaggle free tier is RAM-poor (~13 GB) and VRAM-ok (T4 = 15 GB). The original
# enable_model_cpu_offload() held weights in system RAM and streamed them,
# which killed the kernel on the first real run (2026-07-25). The 1.3B model
# fits the T4 directly in fp16 — keep it resident on the GPU. VAE slicing and
# tiling keep the decode step inside VRAM too.
import gc

import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator = GPU (needs phone verification)"
import diffusers
assert tuple(int(x) for x in diffusers.__version__.split(".")[:2]) >= (0, 33), (
    f"diffusers is {diffusers.__version__} — WanPipeline needs >= 0.33. "
    "Re-run the setup cell, then Run > Restart & clear cell outputs.")
from diffusers import WanPipeline
from diffusers.utils import export_to_video

pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,          # stream shards in; never hold two copies
)
pipe.to("cuda")                       # resident on the T4, not offloaded to RAM
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"pipeline ready on {torch.cuda.get_device_name(0)} — "
      f"{free/2**30:.1f} GB of {total/2**30:.1f} GB VRAM free")


In [ ]:
# ---- generate: 480x832 (9:16), 81 frames @ 16fps = ~5s per beat ------------
from datetime import date
out = pathlib.Path("/kaggle/working/clips"); out.mkdir(parents=True, exist_ok=True)
for s in todo:
    dest = out / f"{s['num']:02d}-{s['slug']}.mp4"
    if dest.exists():
        print(f"skip {dest.name} (exists)"); continue
    print(f"beat {s['num']:02d} ({s['slug']}) …", flush=True)
    g = torch.Generator(device="cpu").manual_seed(SEED + s["num"])
    frames = pipe(prompt=s["prompt"],
                  negative_prompt="photorealistic, 3d render, text, watermark, low quality, blurry",
                  height=832, width=480, num_frames=81,
                  num_inference_steps=STEPS, generator=g).frames[0]
    export_to_video(frames, str(dest), fps=16)
    dest.with_suffix(".meta.yaml").write_text(
        "# Shot provenance (\u00a77.2)\n" + yaml.safe_dump({
            "platform": "kaggle-free-gpu", "model": "Wan2.1-T2V-1.3B (Apache-2.0)",
            "shot_beat": s["num"], "prompt": s["prompt"], "seed": SEED + s["num"],
            "steps": STEPS, "duration_s": 5, "aspect": "9:16 (480x832)",
            "cost_usd": 0.0, "date": date.today().isoformat(),
        }, sort_keys=False, allow_unicode=True))
    print(f"  \u2713 {dest.name}")
    del frames; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ---- pack for download ------------------------------------------------------
import shutil
shutil.make_archive("/kaggle/working/clips", "zip", "/kaggle/working/clips")
print("download clips.zip from the Output tab, then locally:")
print(f"  python3 pipeline/render_t3.py {GENOME} {NODE} --clips <unzipped-dir> --out /tmp/{NODE}-episode.mp4")